In [1]:
# [NEW SETUP CELL - added during reorganization]
# This notebook now lives in notebooks/. Moving up to the project root
# so the original 'data/...' file paths below keep working unchanged.
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")

print("Working directory:", Path.cwd())


Working directory: /Users/adipshrestha/Documents/practicum/Project4_Daylight


In [2]:
import pyreadr

result = pyreadr.read_r('data/fars.RData', encoding='cp1252')
fars = result['fars']
fars.head()

,state,statename,st_case,ve_total,ve_forms,pvh_invl,peds,persons,permvit,pernotmvit,...,cf1,cf1name,cf2,cf2name,cf3,cf3name,fatals,drunk_dr,countyname,cityname
0,1,Alabama,10001,1,1,0,0,1,1,0,...,0,None,0,None,0,None,1,1,NaN,NaN
1,1,Alabama,10002,1,1,0,0,1,1,0,...,0,None,0,None,0,None,1,0,NaN,NaN
2,1,Alabama,10003,1,1,0,0,2,2,0,...,0,None,0,None,0,None,1,1,NaN,NaN
3,1,Alabama,10004,1,1,0,0,1,1,0,...,0,None,0,None,0,None,1,1,NaN,NaN
4,1,Alabama,10005,2,2,0,0,2,2,0,...,0,None,0,None,0,None,1,0,NaN,NaN


In [3]:
import pandas as pd
import matplotlib.pyplot as plt 

Data set over view - 

In [4]:

# Checking the size and time coverage of our dataset

print("Number of crashes:", fars.shape[0])
print("Number of variables:", fars.shape[1])

print("\nColumns available:")
print(fars.columns.tolist())

print("\nYears included:")
print(sorted(fars['year'].dropna().unique()))

Number of crashes: 358460
Number of variables: 91

Columns available:
['state', 'statename', 'st_case', 've_total', 've_forms', 'pvh_invl', 'peds', 'persons', 'permvit', 'pernotmvit', 'county', 'city', 'day', 'dayname', 'month', 'monthname', 'year', 'day_week', 'day_weekname', 'hour', 'hourname', 'minute', 'minutename', 'nhs', 'nhsname', 'route', 'routename', 'tway_id', 'tway_id2', 'rur_urb', 'rur_urbname', 'func_sys', 'func_sysname', 'rd_owner', 'rd_ownername', 'milept', 'mileptname', 'latitude', 'latitudename', 'longitud', 'longitudname', 'sp_jur', 'sp_jurname', 'harm_ev', 'harm_evname', 'man_coll', 'man_collname', 'reljct1', 'reljct1name', 'reljct2', 'reljct2name', 'typ_int', 'typ_intname', 'wrk_zone', 'wrk_zonename', 'rel_road', 'rel_roadname', 'lgt_cond', 'lgt_condname', 'weather1', 'weather1name', 'weather2', 'weather2name', 'weather', 'weathername', 'sch_bus', 'sch_busname', 'rail', 'railname', 'not_hour', 'not_hourname', 'not_min', 'not_minname', 'arr_hour', 'arr_hourname', 'ar

importing data till 2000-

In [5]:

import requests
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO

# Years not already included in our dataset
missing_years = range(2000, 2015)

# Folder for the original downloaded files
data_folder = Path("data")
data_folder.mkdir(exist_ok=True)

for year in missing_years:
    filename = f"FARS{year}NationalCSV.zip"
    filepath = data_folder / filename

    if filepath.exists():
        print(f"{year}: Already downloaded")
        continue

    url = (
        f"https://static.nhtsa.gov/nhtsa/downloads/"
        f"FARS/{year}/National/{filename}"
    )

    try:
        response = requests.get(url, timeout=120)
        response.raise_for_status()

        # Check that the downloaded file is a valid ZIP
        with ZipFile(BytesIO(response.content)) as z:
            if z.testzip() is not None:
                raise ValueError("ZIP integrity check failed")

        filepath.write_bytes(response.content)
        print(f"{year}: Downloaded successfully")

    except Exception as e:
        print(f"{year}: Failed - {e}")


2000: Already downloaded
2001: Already downloaded
2002: Already downloaded
2003: Already downloaded
2004: Already downloaded
2005: Already downloaded
2006: Already downloaded
2007: Already downloaded
2008: Already downloaded
2009: Already downloaded
2010: Already downloaded
2011: Already downloaded
2012: Already downloaded
2013: Already downloaded
2014: Already downloaded


In [6]:
# Inspecting the contents of the 2014 zip file
zip_path = data_folder / "FARS2014NationalCSV.zip"

with ZipFile(zip_path) as z:
    files = z.namelist()

# Printing the files inside
for file in files:
    print(file)

FARS2014NationalCSV/ACCIDENT.csv
FARS2014NationalCSV/CEVENT.CSV
FARS2014NationalCSV/DAMAGE.CSV
FARS2014NationalCSV/DISTRACT.CSV
FARS2014NationalCSV/DRIMPAIR.CSV
FARS2014NationalCSV/FACTOR.CSV
FARS2014NationalCSV/MANEUVER.CSV
FARS2014NationalCSV/MIACC.CSV
FARS2014NationalCSV/MIDRVACC.CSV
FARS2014NationalCSV/MIPER.CSV
FARS2014NationalCSV/NMCRASH.CSV
FARS2014NationalCSV/NMIMPAIR.CSV
FARS2014NationalCSV/NMPRIOR.CSV
FARS2014NationalCSV/PARKWORK.CSV
FARS2014NationalCSV/PBTYPE.CSV
FARS2014NationalCSV/PERSON.CSV
FARS2014NationalCSV/SAFETYEQ.CSV
FARS2014NationalCSV/VEHICLE.CSV
FARS2014NationalCSV/VEVENT.CSV
FARS2014NationalCSV/VINDECODE.CSV
FARS2014NationalCSV/VIOLATN.CSV
FARS2014NationalCSV/VISION.CSV
FARS2014NationalCSV/vpicdecode.csv
FARS2014NationalCSV/VSOE.CSV


In [7]:

# Reading the 2014 accident data
with ZipFile(zip_path) as z:
    accident_file = [f for f in z.namelist()
                     if f.lower().endswith("accident.csv")][0]

    with z.open(accident_file) as file:
        fars_2014 = pd.read_csv(file, encoding="latin1", low_memory=False)

# Looking at the first five rows
print("Shape:", fars_2014.shape)
display(fars_2014.head())


Shape: (30056, 50)


,STATE,ST_CASE,VE_TOTAL,VE_FORMS,PVH_INVL,PEDS,PERNOTMVIT,PERMVIT,PERSONS,COUNTY,...,NOT_MIN,ARR_HOUR,ARR_MIN,HOSP_HR,HOSP_MN,CF1,CF2,CF3,FATALS,DRUNK_DR
0,1,10001,1,1,0,0,0,2,2,71,...,99,1,35,99,99,0,0,0,1,1
1,1,10002,1,1,0,0,0,1,1,59,...,99,13,50,99,99,0,0,0,1,1
2,1,10003,2,2,0,0,0,7,7,125,...,99,3,10,99,99,0,0,0,2,0
3,1,10004,3,3,0,0,0,5,5,121,...,99,9,15,99,99,0,0,0,1,0
4,1,10005,1,1,0,0,0,1,1,43,...,99,12,45,88,88,0,0,0,1,0


In [8]:

# Loading accident data from 2000 to 2014
old_data = []

for year in range(2000, 2015):
    zip_path = data_folder / f"FARS{year}NationalCSV.zip"

    with ZipFile(zip_path) as z:
        accident_file = [f for f in z.namelist()
                         if f.lower().endswith("accident.csv")][0]

        with z.open(accident_file) as file:
            df = pd.read_csv(file, encoding="latin1", low_memory=False)

    # Making column names consistent
    df.columns = df.columns.str.lower()

    # Keeping track of the year
    df["year"] = year

    old_data.append(df)
    print(f"{year}: {len(df):,} crashes")

# Combining all 15 years
fars_old = pd.concat(old_data, ignore_index=True)

print("\nTotal historical crashes:", len(fars_old))
print("Number of columns:", fars_old.shape[1])


2000: 37,526 crashes
2001: 37,862 crashes
2002: 38,491 crashes
2003: 38,477 crashes
2004: 38,444 crashes
2005: 39,252 crashes
2006: 38,648 crashes
2007: 37,435 crashes
2008: 34,172 crashes
2009: 30,862 crashes
2010: 30,296 crashes
2011: 29,867 crashes
2012: 31,006 crashes
2013: 30,202 crashes
2014: 30,056 crashes

Total historical crashes: 522596
Number of columns: 62


### Combining our 25 years of data

In [9]:

# Combining the old and new datasets
fars_full = pd.concat([fars_old, fars], ignore_index=True)

# Checking the total number of crashes and years
print("Total crashes:", len(fars_full))
print("Total columns:", fars_full.shape[1])
print("Years:", sorted(fars_full["year"].unique()))

# Checking for duplicate crash records
duplicates = fars_full.duplicated(
    subset=["year", "state", "st_case"]
).sum()

print("Duplicate crashes:", duplicates)

# Checking the number of crashes per year
display(fars_full.groupby("year").size())


Total crashes: 881056
Total columns: 104
Years: [2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Duplicate crashes: 0


year
2000    37526
2001    37862
2002    38491
2003    38477
2004    38444
2005    39252
2006    38648
2007    37435
2008    34172
2009    30862
2010    30296
2011    29867
2012    31006
2013    30202
2014    30056
2015    32538
2016    34748
2017    34560
2018    33919
2019    33487
2020    35935
2021    39785
2022    39422
2023    37769
2024    36297
dtype: int64

In [10]:
# [NEW CELL - added during reorganization]
# Saving the combined 2000-2024 dataset so teammates can load it directly
output_path = Path("data") / "fars_2000_2024.parquet"
fars_full.to_parquet(output_path, index=False)

# Reading the file back to check that the complete dataset was saved
check = pd.read_parquet(output_path)
print("Saved to:", output_path)
print("Rows:", len(check))
print("Columns:", check.shape[1])
print("Years:", check["year"].min(), "to", check["year"].max())
print("Duplicate crashes:",
      check.duplicated(subset=["year", "state", "st_case"]).sum())


Saved to: data/fars_2000_2024.parquet
Rows: 881056
Columns: 104
Years: 2000 to 2024
Duplicate crashes: 0
